# Cambridge Economics Grader — Data Explorer
Use this notebook to understand your training data before training the model.
Run each cell top to bottom with **Shift + Enter**.

In [ ]:
# Cell 1 — Setup: run this first every time you open the notebook
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Make sure src/ is on the path
ROOT = Path().resolve().parent
sys.path.append(str(ROOT / 'src'))

from config import RAW_ESSAYS_DIR, TRAINING_FILE, EVAL_FILE
from feedback_engine import EssayAnalyser

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
print('✓ Setup complete')

---
## 1. Load Your Essays

In [ ]:
# Cell 2 — Load the essays CSV
csv_path = RAW_ESSAYS_DIR / 'essays.csv'

if not csv_path.exists():
    print('⚠  No essays.csv found yet.')
    print('   Add essays first: python scripts/add_essay.py')
    df = pd.DataFrame()
else:
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    print(f'✓ Loaded {len(df)} essays')
    print(f'  Columns: {list(df.columns)}')
    df.head(3)

In [ ]:
# Cell 3 — Basic dataset summary
if not df.empty:
    print('=== Dataset Summary ===')
    print(f'Total essays      : {len(df)}')
    print(f'AS Level essays   : {len(df[df.level=="AS"])}')
    print(f'IGCSE essays      : {len(df[df.level=="IGCSE"])}')
    print(f'Has feedback      : {df.feedback.notna().sum()} essays')
    print()
    print('Mark distribution:')
    print(df.groupby(['level','mark']).size().to_string())

---
## 2. Mark Distribution

In [ ]:
# Cell 4 — Mark distribution histogram
if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, (level, max_m) in zip(axes, [('AS', 12), ('IGCSE', 8)]):
        subset = df[df.level == level]
        if subset.empty:
            ax.set_title(f'{level} — no data yet')
            continue
        counts = subset['mark'].value_counts().sort_index()
        bars = ax.bar(counts.index, counts.values, color='steelblue', edgecolor='navy', alpha=0.8)
        ax.set_title(f'{level} Mark Distribution (out of {max_m})', fontsize=13)
        ax.set_xlabel('Mark')
        ax.set_ylabel('Number of Essays')
        ax.set_xticks(range(0, max_m + 1))
        # Label each bar
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, str(int(h)),
                        ha='center', va='bottom', fontsize=9)

    plt.suptitle('How marks are spread across your training data', y=1.02)
    plt.tight_layout()
    plt.show()
    print('TIP: Aim for at least 2–3 essays per mark band for balanced training.')

---
## 3. Essay Length vs Mark

In [ ]:
# Cell 5 — Word count analysis
if not df.empty:
    df['word_count'] = df['essay'].astype(str).apply(lambda x: len(x.split()))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for ax, level in zip(axes, ['AS', 'IGCSE']):
        subset = df[df.level == level]
        if subset.empty:
            ax.set_title(f'{level} — no data')
            continue
        ax.scatter(subset['mark'], subset['word_count'],
                   alpha=0.6, s=70, color='steelblue', edgecolors='navy')
        # Trend line
        if len(subset) >= 3:
            z = np.polyfit(subset['mark'], subset['word_count'], 1)
            p = np.poly1d(z)
            x_line = np.linspace(subset['mark'].min(), subset['mark'].max(), 100)
            ax.plot(x_line, p(x_line), 'r--', alpha=0.6, label='Trend')
            ax.legend()
        ax.set_title(f'{level}: Word Count vs Mark')
        ax.set_xlabel('Mark')
        ax.set_ylabel('Word Count')

    plt.suptitle('Do longer essays score higher?')
    plt.tight_layout()
    plt.show()

    print('Average word counts by mark band:')
    print(df.groupby('level')['word_count'].describe().round(0).to_string())

---
## 4. Feature Analysis — What Distinguishes High vs Low Scorers?

In [ ]:
# Cell 6 — Run the rule-based analyser on every essay
if not df.empty:
    print('Analysing essays for examiner keywords... (may take a moment)')

    records = []
    for _, row in df.iterrows():
        try:
            analyser  = EssayAnalyser(str(row['essay']), str(row['level']),
                                       int(row['max_marks']))
            checklist = analyser.get_checklist()
            records.append({
                'level'          : row['level'],
                'mark'           : row['mark'],
                'max_marks'      : row['max_marks'],
                'word_count'     : checklist['word_count'],
                'eval_words'     : checklist['eval_word_count'],
                'analysis_phrases': checklist['analysis_count'],
                'econ_concepts'  : checklist['concept_count'],
                'has_diagram'    : int(checklist['has_diagram']),
                'has_example'    : int(checklist['has_real_example']),
                'has_conclusion' : int(checklist['has_conclusion']),
            })
        except Exception as e:
            pass

    features_df = pd.DataFrame(records)
    print(f'✓ Analysed {len(features_df)} essays')
    features_df.head()

In [ ]:
# Cell 7 — Correlation heatmap: which features correlate with higher marks?
if 'features_df' in dir() and not features_df.empty:
    numeric_cols = ['mark', 'word_count', 'eval_words', 'analysis_phrases',
                    'econ_concepts', 'has_diagram', 'has_example', 'has_conclusion']
    corr = features_df[numeric_cols].corr()

    plt.figure(figsize=(9, 7))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
                center=0, mask=mask, square=True,
                linewidths=0.5, cbar_kws={'label': 'Correlation'})
    plt.title('Feature Correlations with Mark\n(green = positive, red = negative)', pad=15)
    plt.tight_layout()
    plt.show()

    print('\nTop features correlated with mark:')
    corr_with_mark = corr['mark'].drop('mark').sort_values(ascending=False)
    for feat, val in corr_with_mark.items():
        bar = '█' * int(abs(val) * 20)
        sign = '+' if val >= 0 else '-'
        print(f'  {feat:<22} {sign}{abs(val):.3f}  {bar}')

In [ ]:
# Cell 8 — High scorers vs low scorers: feature comparison
if 'features_df' in dir() and not features_df.empty:
    for level in features_df['level'].unique():
        subset   = features_df[features_df.level == level].copy()
        max_m    = subset['max_marks'].iloc[0]
        midpoint = max_m * 0.6

        high = subset[subset.mark >= midpoint]
        low  = subset[subset.mark <  midpoint]

        if high.empty or low.empty:
            print(f'{level}: not enough data to compare bands (need essays in both high and low bands)')
            continue

        feats = ['word_count', 'eval_words', 'analysis_phrases',
                 'econ_concepts', 'has_diagram', 'has_example', 'has_conclusion']

        x      = np.arange(len(feats))
        width  = 0.35
        fig, ax = plt.subplots(figsize=(11, 5))

        bars_high = ax.bar(x - width/2, high[feats].mean(), width,
                           label=f'High scorers (≥{int(midpoint)}/{max_m})',
                           color='seagreen', alpha=0.85, edgecolor='darkgreen')
        bars_low  = ax.bar(x + width/2, low[feats].mean(),  width,
                           label=f'Low scorers (<{int(midpoint)}/{max_m})',
                           color='tomato',   alpha=0.85, edgecolor='darkred')

        ax.set_xticks(x)
        ax.set_xticklabels([f.replace('_', ' ').title() for f in feats], rotation=25, ha='right')
        ax.set_ylabel('Average Count / Proportion')
        ax.set_title(f'{level} — What separates high scorers from low scorers?')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f'  High scorers: n={len(high)}   Low scorers: n={len(low)}')

---
## 5. Topic Coverage

In [ ]:
# Cell 9 — Which topics are covered in your training data?
if not df.empty and 'topic' in df.columns:
    topic_counts = df['topic'].fillna('(no tag)').value_counts()

    fig, ax = plt.subplots(figsize=(10, max(4, len(topic_counts) * 0.4)))
    colors  = plt.cm.Set2(np.linspace(0, 1, len(topic_counts)))
    bars    = ax.barh(topic_counts.index, topic_counts.values,
                      color=colors, edgecolor='grey', alpha=0.85)

    for bar in bars:
        w = bar.get_width()
        ax.text(w + 0.05, bar.get_y() + bar.get_height()/2,
                str(int(w)), va='center', fontsize=9)

    ax.set_xlabel('Number of Essays')
    ax.set_title('Essays per Topic')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    print('TIP: Try to have essays covering all major Cambridge topics:')
    print('  AS:    Macroeconomics, Microeconomics, Market Failure, International')
    print('  IGCSE: Allocation, Price System, Market Failure, Macroeconomics, International')

---
## 6. Training File Inspector

In [ ]:
# Cell 10 — Inspect the processed training JSONL (after running data_prep.py)
if TRAINING_FILE.exists():
    with open(TRAINING_FILE, encoding='utf-8') as f:
        samples = [json.loads(line) for line in f if line.strip()]

    print(f'Training samples: {len(samples)}')

    if EVAL_FILE.exists():
        with open(EVAL_FILE, encoding='utf-8') as f:
            eval_samples = [json.loads(line) for line in f if line.strip()]
        print(f'Eval samples    : {len(eval_samples)}')

    print('\n--- First training sample (user message preview) ---')
    first = samples[0]
    for msg in first['messages']:
        role    = msg['role'].upper()
        preview = msg['content'][:300].replace('\n', ' ')
        print(f'[{role}]: {preview}...')
        print()
else:
    print('⚠  Training file not found.')
    print('   Run: python src/data_prep.py  to generate it.')

---
## 7. Data Quality Check

In [ ]:
# Cell 11 — Check for data quality issues before training
if not df.empty:
    issues = []

    # Check for very short essays
    df['wc'] = df['essay'].astype(str).apply(lambda x: len(x.split()))
    short = df[df['wc'] < 80]
    if not short.empty:
        issues.append(f'⚠  {len(short)} essay(s) have fewer than 80 words — check these rows: {list(short.index)}')

    # Check mark ranges
    bad_marks = df[df.apply(lambda r: r['mark'] > r['max_marks'], axis=1)]
    if not bad_marks.empty:
        issues.append(f'⛔ {len(bad_marks)} essay(s) have mark > max_marks — fix these rows: {list(bad_marks.index)}')

    # Check level values
    bad_levels = df[~df['level'].str.upper().isin(['AS', 'IGCSE'])]
    if not bad_levels.empty:
        issues.append(f'⛔ {len(bad_levels)} essay(s) have invalid level value — must be AS or IGCSE')

    # Check balance
    for level in ['AS', 'IGCSE']:
        subset   = df[df.level == level]
        if subset.empty:
            continue
        max_m    = subset['max_marks'].iloc[0]
        for mark in range(0, max_m + 1):
            count = len(subset[subset.mark == mark])
            if count == 0:
                issues.append(f'ℹ  No {level} essay with mark {mark}/{max_m} — consider adding one for balanced training')

    if issues:
        print('Data Quality Report:')
        for issue in issues:
            print(' ', issue)
    else:
        print('✓ No data quality issues found. Ready to train!')

    print(f'\nTotal essays ready: {len(df)}')
    print('Recommendation: 30+ essays gives good training results, 60+ gives strong results.')

---
## 8. Quick Essay Tester (no model needed)

In [ ]:
# Cell 12 — Paste any essay here to run the instant rule-based analysis
# Change the values below and re-run this cell

MY_ESSAY = """
Paste your essay here.
You can write multiple paragraphs.
Replace this entire block with your real essay text.
"""

MY_LEVEL     = 'AS'    # 'AS' or 'IGCSE'
MY_MAX_MARKS = 12      # 12 for AS, 8 for IGCSE

# Run the analysis
from feedback_engine import show_essay_analysis
show_essay_analysis(MY_ESSAY.strip(), MY_LEVEL, MY_MAX_MARKS)